# Capítulo 7 — Robustez certificada: garantias, não promessas

## Missão
Treino adversarial resiste aos ataques que **você testou**. E se aparecer um ataque melhor? O cap. 10 do livro fala de **verificação certificada**: provar que, dentro de um raio, **nenhuma** perturbação muda a resposta. Você vai:
- calcular a garantia **exata** de um classificador linear (a distância até a fronteira);
- implementar a **suavização aleatória** (*randomized smoothing*), que certifica qualquer modelo com um raio L2 a partir de muitas previsões com ruído gaussiano.

**No fim deste caderno você terá:** o módulo `meu_escudo/certificado.py` (raio exato do modelo linear e suavização aleatória com o raio certificado de Cohen et al.) e a curva de acurácia certificada.

**Leitura essencial (opcional, livro em inglês):**
- 10.2 Background · PDF p. 153
- 10.3 Certificated Verification For Adversarial Examples · PDF p. 156
- 10.4 Robustness On Adversarial State Observations · PDF p. 159

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap07_robustez_certificada_garantias_nao_promessas.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_escudo/
os.makedirs("../meu_escudo", exist_ok=True)
open("../meu_escudo/__init__.py", "a").close()        # meu_escudo/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_escudo")

## Ideia em 30 segundos
**Linear:** a classe `c` vence a classe `j` enquanto `(w_c - w_j)·x + (b_c - b_j) > 0`. A menor perturbação L2 que empata é essa margem dividida por `‖w_c - w_j‖`. O raio certificado é o mínimo sobre os `j`. É exato.

**Suavização aleatória:**
- o classificador suavizado `g(x)` responde a classe mais votada de `f(x + ruído)`, com o ruído `N(0, σ²)`;
- se a classe vencedora sai com probabilidade `p_A > 1/2`, então `g` não muda dentro do raio `R = σ·Φ⁻¹(p_A)`;
- `p_A` vem de uma estimativa conservadora (o limite inferior de Clopper-Pearson).

### 7.1 **As garantias.** Crie `certificado.py` com:
- `raio_linear(W, b, x)`: devolve `(classe, raio)` exatos, em L2;
- `suavizado(prever, x, sigma, n=500, semente=0)`: a contagem de votos com ruído;
- `certificar(prever, x, sigma, n0=100, n=1000, alfa=0.001, semente=0)`: escolhe a classe com `n0` amostras, estima `p_A` com outras `n` (limite inferior de Clopper-Pearson via `scipy.stats.beta.ppf`) e devolve `(classe, raio)` ou `(None, 0)` se `p_A ≤ 0,5`, quando o modelo se abstém.

_A célula grava `meu_escudo/certificado.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_escudo/certificado.py


### 7.2 **Raio exato do linear.** Treine a `LogisticRegression` nos dígitos e calcule o raio certificado de 300 exemplos do teste. Mostre a mediana e a fração com raio maior que 0,5. Depois confira um caso: aplique um ataque L2 na direção ótima com 99% do raio (a previsão não muda) e com 101% (muda).

### 7.3 **Suavização aleatória num modelo qualquer.** Treine um `MLPClassifier((128,))` **com ruído** (os dados de treino repetidos 3 vezes com `N(0, 0,25²)`), o que é essencial para a suavização funcionar. Certifique 100 exemplos do teste com σ = 0,25 e mostre a fração certificada, a acurácia certificada (certificou **e** acertou) e o raio mediano dos certificados.

### 7.4 **Curva de acurácia certificada.** Para os raios R de 0 a 1,2, desenhe a fração de exemplos com a classe certa **e** raio certificado ≥ R, para o MLP suavizado e para o modelo linear (raio exato). Qual oferece mais garantia?

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
